# 02 - Expose the skill through Toolbox

Attach the skill imported in notebook 01 to a skills-only Foundry Toolbox. Verify its instructions are available through the MCP endpoint for a compatible MAF client to consume later.

These cells make real service calls. Use the same Foundry project as notebook 01. The agent integration is the next notebook; retrieving instructions here does not execute them.

## A Toolbox version is a configuration snapshot

Skills and Toolboxes have separate versions. Each Toolbox version can select a particular version of each skill:

```text
Toolbox version 1
|-- Skill A -> version 2
`-- Skill B -> version 3

Toolbox version 2
|-- Skill A -> version 4
`-- Skill B -> version 3
```

**Publishing Toolbox version 2 means: use this configuration by default.**

- Agents using the **default Toolbox endpoint** get configuration 2 when they fetch it. Clients that cache configuration may need to reload.
- Agents explicitly requesting **Toolbox version 1** keep configuration 1.
- Uploading **Skill A version 5** does not change either configuration: both select a specific skill version. This is called **pinning**.

You can test and release a combination of skill versions together, then roll back by selecting an earlier Toolbox version as default. The example uses two skills to illustrate the relationship; this notebook attaches one.

## Configure the project and skill reference

Load the workspace-root .env. Copy FOUNDRY_SKILL_NAME and FOUNDRY_SKILL_VERSION from notebook 01 into that file first. FOUNDRY_TOOLBOX_NAME selects the Toolbox to create. Skill and Toolbox version identifiers are independent. Restart the kernel after editing .env; existing process variables take precedence.

In [ ]:
import os
import re
from pathlib import Path
from urllib.parse import urlsplit

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ToolboxSkillReference
from azure.identity import AzureCliCredential
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from httpx2 import AsyncClient
from mcp.types import PaginatedRequestParams, TextResourceContents

ROOT = Path.cwd().resolve()
if not (ROOT / "requirements.txt").is_file():
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").is_file() or not (ROOT / ".env.example").is_file():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")
load_dotenv(ROOT / ".env", override=False, interpolate=False)

# Azure CLI accepts a subscription or tenant selector, not both.
subscription = os.environ.get("AZURE_SUBSCRIPTION_ID", "").strip()
tenant = os.environ.get("AZURE_TENANT_ID", "").strip()
CREDENTIAL_OPTIONS = {"subscription": subscription} if subscription else ({"tenant_id": tenant} if tenant else {})
PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "").rstrip("/")
endpoint = urlsplit(PROJECT_ENDPOINT)
if (
    endpoint.scheme != "https" or not endpoint.hostname
    or not endpoint.hostname.endswith(".services.ai.azure.com")
    or endpoint.username is not None or endpoint.password is not None
    or endpoint.port not in (None, 443) or endpoint.query or endpoint.fragment
    or not re.fullmatch(r"/api/projects/[^/]+", endpoint.path)
):
    raise ValueError("Set FOUNDRY_PROJECT_ENDPOINT to https://<resource>.services.ai.azure.com/api/projects/<project>.")

SKILL_NAME = os.environ.get("FOUNDRY_SKILL_NAME", "engineering-impact-assessment")
SKILL_VERSION = os.environ.get("FOUNDRY_SKILL_VERSION", "")
if not re.fullmatch(r"[A-Za-z0-9._-]+", SKILL_VERSION):
    raise ValueError("Set FOUNDRY_SKILL_VERSION to the version printed by notebook 01.")
TOOLBOX_NAME = os.environ.get("FOUNDRY_TOOLBOX_NAME", "engineering-skills-demo").strip()

## Create a Toolbox version

Notebook 01 uploaded your skill to Foundry. This step adds that existing skill to a Toolbox so a MAF agent can retrieve it later. There is no second upload.

SKILL_VERSION selects the exact skill version to use. For example, if you select skill version `2`, the Toolbox keeps using `2` even after you upload skill version `3`.

The Toolbox also has its own versions, separate from the skill's versions. Running the cell creates a new Toolbox version. The first is selected automatically for agents using the default Toolbox endpoint. For later versions, the publish cell below lets you choose when to switch that endpoint to the new version.

In [ ]:
# Pin skill content independently of the Toolbox configuration version.
skill_reference = ToolboxSkillReference(name=SKILL_NAME, version=SKILL_VERSION)
with AzureCliCredential(**CREDENTIAL_OPTIONS) as credential, AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True, headers={"Foundry-Features": "Skills=V1Preview"}) as project:
    project.beta.skills.get_version(SKILL_NAME, SKILL_VERSION)
    toolbox_version = project.toolboxes.create_version(name=TOOLBOX_NAME, description="Shared skills for MAF agents.", tools=[], skills=[skill_reference])
    print(toolbox_version.as_dict())

TOOLBOX_VERSION = toolbox_version.version
mcp_endpoint = f"{PROJECT_ENDPOINT}/toolboxes/{TOOLBOX_NAME}/versions/{TOOLBOX_VERSION}/mcp?api-version=v1"
print("Candidate MCP endpoint:", mcp_endpoint)

## Inspect the candidate through MCP

Initialize a session, list skill resources and read SKILL.md. Discovery follows pagination cursors. This uses legacy MCP resource discovery, not the MCP Skills extension.

Reading a resource does not activate a skill in an agent. The later MAF integration must discover the skill, load it when relevant and apply its instructions.

In [ ]:
async def inspect_mcp(endpoint: str) -> None:
    with AzureCliCredential(**CREDENTIAL_OPTIONS) as credential:
        token = credential.get_token("https://ai.azure.com/.default")
        headers = {"Authorization": "Bearer " + token.token}
        async with AsyncClient(headers=headers) as http_client, streamable_http_client(endpoint, http_client=http_client) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                initialized = await session.initialize()
                print("Capabilities:", initialized.capabilities.model_dump(mode="json", exclude_none=True))
                resources = []
                cursor = None
                while True:
                    page = await session.list_resources(params=PaginatedRequestParams(cursor=cursor))
                    resources.extend(page.resources)
                    cursor = page.next_cursor
                    if not cursor:
                        break
                print("Resources:", [str(resource.uri) for resource in resources])
                expected_uri = f"skill://{SKILL_NAME}/SKILL.md"
                matches = [resource for resource in resources if str(resource.uri) == expected_uri]
                if len(matches) != 1:
                    raise RuntimeError("Expected one skill resource. Check the version, permissions and Toolbox skill discovery support.")
                # Skill instructions are resource content, not a tool result.
                loaded = await session.read_resource(matches[0].uri)
                content = "\n".join(item.text for item in loaded.contents if isinstance(item, TextResourceContents))
                if not content.strip():
                    raise RuntimeError("The skill resource returned no readable instructions.")
                print(content)

await inspect_mcp(mcp_endpoint)

## Publish the inspected Toolbox version

**Publish** selects the default configuration served by the unversioned consumer endpoint. It does not make the endpoint public, grant access or deploy an agent. Run after reviewing the candidate above.

Version-pinned clients remain unchanged. Default-following clients may need to reload cached configuration. Rollback uses the same operation with an older retained identifier as SELECTED_VERSION.

In [ ]:
SELECTED_VERSION = TOOLBOX_VERSION
with AzureCliCredential(**CREDENTIAL_OPTIONS) as credential, AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
    project.toolboxes.get_version(TOOLBOX_NAME, SELECTED_VERSION)
    published = project.toolboxes.update(name=TOOLBOX_NAME, default_version=SELECTED_VERSION)
    assert published.default_version == SELECTED_VERSION
    print("Default version:", published.default_version)

consumer_endpoint = f"{PROJECT_ENDPOINT}/toolboxes/{TOOLBOX_NAME}/mcp?api-version=v1"
print("Consumer MCP endpoint:", consumer_endpoint)

## Verify in the Foundry portal

After running the publish cell above, open [Microsoft Foundry](https://ai.azure.com) and select the same project as PROJECT_ENDPOINT. Go to **Tools > Toolboxes**. Find TOOLBOX_NAME and compare the displayed version with the default version printed by the publish cell.

This example shows `engineering-skills-demo` at Toolbox version **3**. It is the configuration version, not the skill version: this configuration references skill version **2**. Use the identifiers returned by your run; rerunning create cells produces additional versions.

![Foundry Tools > Toolboxes showing engineering-skills-demo version 3](images/foundry-toolboxes.png)

The portal confirms the Toolbox is present. The MCP inspection above verifies its skill content is readable; notebook 03 verifies that a MAF agent actually loads and applies it.

## Next: build the agent

[Notebook 03](03_maf_toolbox_agent.ipynb) connects a Microsoft Agent Framework (MAF) agent to this endpoint, loads the skill on demand and checks its effect on a completed task. Hosting and scaling follow separately.

[Skills](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/skills) | [Toolbox](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/toolbox)